In [1]:
pip install sounddevice numpy librosa openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 3.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 2.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 694.4 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 3.1 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 2.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 3.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 3.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=cfd38aee8cb2b065b86ffb4168a8f3aeaeaeb81935fef37d4ceb34afff64d35f
  Sto

In [7]:
pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 2.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 2.2 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 3.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [faster-whisper]m [huggingface-hub]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install --upgrade certifi

  Attempting uninstall: certifi
    Found existing installation: certifi 2026.1.4
    Uninstalling certifi-2026.1.4:
      Successfully uninstalled certifi-2026.1.4

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import sounddevice as sd
import numpy as np
import queue
import threading
import librosa
from faster_whisper import WhisperModel
import time

model = WhisperModel("base", compute_type="int8")
audio_queue = queue.Queue()

SAMPLE_RATE = 16000
WINDOW_DURATION = 5  # seconds

voice_metrics = {
    "wpm": 0,
    "volume_stability": 0,
    "filler_count": 0
}

def audio_callback(indata, frames, time_info, status):
    audio_queue.put(indata.copy())

def analyze_audio_chunk(audio_chunk):
    global voice_metrics

    # Flatten audio
    y = audio_chunk.flatten()

    # Volume stability
    rms = librosa.feature.rms(y=y)[0]
    mean_volume = np.mean(rms)
    volume_std = np.std(rms)
    stability = 1 - (volume_std / (mean_volume + 1e-6))

    # Transcription
    segments, _ = model.transcribe(y, beam_size=1)

    text = ""
    for segment in segments:
        text += segment.text

    words = len(text.split())
    wpm = (words / WINDOW_DURATION) * 60

    fillers = ["um", "uh", "like", "basically", "actually"]
    filler_count = sum(text.lower().count(f) for f in fillers)

    voice_metrics["wpm"] = wpm
    voice_metrics["volume_stability"] = max(0, min(stability, 1))
    voice_metrics["filler_count"] = filler_count
    print("\n--- Voice Update ---")
    print(f"WPM: {wpm:.0f}")
    print(f"Stability: {stability*100:.1f}%")
    print(f"Fillers: {filler_count}")
    print("--------------------")

def voice_thread():
    buffer = []
    start_time = time.time()

    with sd.InputStream(callback=audio_callback,
                        channels=1,
                        samplerate=SAMPLE_RATE):
        while True:
            data = audio_queue.get()
            buffer.append(data)

            if time.time() - start_time >= WINDOW_DURATION:
                audio_chunk = np.concatenate(buffer)
                analyze_audio_chunk(audio_chunk)

                buffer = []
                start_time = time.time()

def start_voice_analysis():
    thread = threading.Thread(target=voice_thread)
    thread.daemon = True
    thread.start()

KeyboardInterrupt: 

In [1]:
pip install fastapi uvicorn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install axios

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 774.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [axios]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip uninstall av

Found existing installation: av 16.1.0
Uninstalling av-16.1.0:
  Would remove:
    /Library/Frameworks/Python.framework/Versions/3.13/bin/pyav
    /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/av-16.1.0.dist-info/*
    /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/av/*
Proceed (Y/n)? ^C
Note: you may need to restart the kernel to use updated packages.


In [8]:
brew --version

NameError: name 'brew' is not defined